# Big B-Router  
Edit out roads that are in front of ALPRs (in a pbf file)  
Follow the ⭐ instructions before runnning each cell in order.  

# 1.  Import libraries and define small functions  
  
⭐ Needed python packages are:   
  
<b>├── geopandas  
  ├── folium   
  ├── pyrosm   
  └── pyosmium   
  </b>  


In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely as shp
from shapely.geometry import Polygon, Point
from pyproj import Geod

import osmium as omi
from osmium import FileProcessor as FProc
from pyrosm import OSM

import warnings
import re

pd.set_option('display.max_columns', None)

g = Geod(ellps="WGS84")
# Function to parse direction,camera:direction tags
def parsedir(strdirection):
    if pd.isna(strdirection):
        return []
    cleaned_string = re.sub(r"[a-zA-Z\.'\" !]", "", str(strdirection)).replace("-", ";").replace(",", ";").replace(":", ";").replace("|", ";")
    directions = [int(d) for d in cleaned_string.split(";") if d and d.isdigit()]
    return directions

# Function to generate trapezoids (ALPR "sightline")
def trapezoidshp(startlng, startlat, direction, height, angle, left_bias, base_width) -> Polygon:
    direction-=left_bias
    hypotenuse = height / np.cos(np.radians((angle/2)))
    bearings = np.array([direction+90, direction-90,direction - angle/2, direction + angle/2])
    lng, lat, _ = g.fwd([startlng]*4, [startlat]*4, bearings, [base_width/2]*2+[hypotenuse]*2)
    return Polygon([(lng[0], lat[0]),(lng[1], lat[1]), (lng[2], lat[2]), (lng[3], lat[3])])

# Function to split "other_tags" into their own columns
def regparse_othertags(row):
    regpat = r'"([^"]+)"=>"([^"]+)"'
    matches = re.findall(regpat, row)
    result = {key: value for key, value in matches}
    return pd.Series(result)

# 2. Load in ALPR data  
⭐ Replace the filenames (filefullarea is not the state file, but the cropped area)  
Modifications like restricting to flock ALPR's only can be made here w/ geopandas/pandas.  

In [2]:
filefullarea = "./pbf/atlanta260526.osm.pbf"
filesurv = "./pbf/atlanta260526-surveillance.osm.pbf"
fileroad = "./pbf/atlanta260526-roads.osm.pbf"



surveillance = gpd.read_file(filesurv, layer="points")
surveillance = surveillance.fillna("")
surveillance = surveillance.drop("other_tags",axis=1).join(surveillance["other_tags"].apply(regparse_othertags))
alpr = surveillance[surveillance["surveillance:type"]=="ALPR"].reset_index(drop=True)
alpr = alpr.rename(columns={"direction":"camdir","camera:direction":"camdir2"})
alpr = alpr.iloc[:,:10].merge(alpr.loc[:,["camdir","camdir2"]], left_index=True, right_index=True)
alpr.to_crs("EPSG:4326", inplace=True)
alpr["direction1"] = alpr["camdir"].apply(parsedir)
alpr["direction2"] = alpr["camdir2"].apply(parsedir)

# 3. Project ALPR data  
⭐ Adjust the parameters or run w/ defaults  
Default is trapezoidal projection.      
Parameters  
  ├── angle - The angle of vision to project from the camera   
  ├── visrange - The (meters) distance away from the camera to project to (this becomes the radius for directionless ALPRs)  
  ├── base_width - Trapezoid base (on position of camera) width in meters  
  └── left_bias - The degrees to tilt direction to the left as cameras tend to be mounted to the right of roads  

In [ ]:
# Trapezoid projection
angle = 45
visrange = 30
base_width = 10
left_bias = 0

shapelist = []
directionless = gpd.GeoDataFrame()

for n, cam in alpr.iterrows():
    directions = []
    if cam["direction1"]:
        directions = cam["direction1"]
    elif cam["direction2"]:
        directions = cam["direction2"]
    if any(x > 360 for x in directions):
        directions = []

    if directions:
        for direction in directions:
            shape = trapezoidshp(cam.geometry.x, cam.geometry.y, direction, visrange, angle, left_bias, base_width)
            shapelist.append({"camindex":n, "direction":direction, "osmid": cam.osm_id, "geometry": shape})
    else:
        circ = gpd.GeoSeries([Point(cam.geometry.x, cam.geometry.y)])
        buffcirc = gpd.GeoSeries(circ, crs="EPSG:4326").to_crs("EPSG:3857").buffer(visrange, resolution=4).to_crs("EPSG:4326")
        shapelist.append({"camindex":n, "osmid":cam.osm_id, "geometry":buffcirc.iloc[0]})

alprtrap = gpd.GeoDataFrame(shapelist, crs="EPSG:4326")
alprtrap = alprtrap.drop_duplicates().reset_index(drop=True)

# 4. Load in roads and get difference from ALPR projection  
If using a different ALPR projection replace the second value in gpd.overlay()  
This cell takes a bit, coffee break?  

In [ ]:
roads = gpd.read_file(fileroad, layer="lines")
roads["geometry"] = roads.geometry.line_merge()
roads = roads.explode().reset_index(drop=True)
roads["geo2"] = roads.geometry
diffedroads = gpd.overlay(roads, alprtrap, how="difference")
diff = diffedroads[diffedroads.geom_type=="LineString"]
diff = diffedroads[diffedroads.geometry==diffedroads.geo2]

# 5. Determine which geometries to drop/edit  

In [7]:
todrop = roads.set_index("osm_id").drop(diffedroads.osm_id)
idstodrop = todrop.index
toedit = diffedroads.drop(diff.index)
explodedit = toedit.explode()
# Get rid of id's with short length (<50m)
explodedit = toedit.explode()
explodedit["length"] = explodedit.to_crs("EPSG:3857").geometry.length
b = explodedit.groupby("osm_id")["length"].sum()
shortsplits = b[b<50].index

idstodrop = np.concatenate((idstodrop, shortsplits)).astype(int)
toedit = toedit.set_index("osm_id").drop(shortsplits)
idstoedit = np.concatenate([toedit.index]).astype(int)

# 6. Drop/Edit geometries and write to PBF file  
⭐ Adjust output filename if desired  

In [ ]:
outputfile = "./bigB.osm.pbf"


fakeid = (2**1)
explodict = explodedit.reset_index(drop=True).groupby("osm_id")
editids = set(idstoedit)
dropids = set(idstodrop)

bothfilter = omi.filter.IdFilter(editids|dropids)
roadsfilter = omi.filter.KeyFilter("highway")

with omi.SimpleWriter(outputfile, overwrite=True) as writer:
    ways = FProc(filefullarea, omi.osm.WAY|omi.osm.NODE)\
            .with_locations()\
            .with_filter(roadsfilter)\
            .with_filter(bothfilter)\
            .with_filter(omi.filter.GeoInterfaceFilter())\
            .handler_for_filtered(writer)
    for obj in ways:
        if obj.is_way():
            if obj.id in dropids:
                pass
            elif obj.id in editids:
                geom = shp.geometry.shape(obj.__geo_interface__["geometry"])
                coords = [(lon, lat) for lon, lat in zip(*geom.xy)]
                clon, clat = zip(*coords)
                refs = [nod.ref for nod in obj.nodes]
                for _id, row in explodict.get_group(str(obj.id)).iterrows():
                    newwaynodes = []
                    newcoords = [(lon, lat) for lon, lat in zip(*row.geometry.xy)]
                    for newlon, newlat in newcoords:
                        _, _, distances = g.inv([newlon]*len(clon), [newlat]*len(clat), clon, clat)
                        closest = min(distances)
                        if closest < 0.01:
                            newwaynodes.append(refs[distances.index(closest)])
                    fakeid+=2
                    writer.add_way(omi.osm.mutable.Way(
                            id=fakeid, nodes=newwaynodes,
                            tags=obj.tags))
                    fakeid+=2
    writer.close()

⭐ The pbf is created! All that's left is to plug it into [OsmAndMapCreator](https://wiki.openstreetmap.org/wiki/OsmAndMapCreator) (Takes a while and uses up a decent amount of ram)  